# 📡 Project 3.3 — PWM Signal Decoder
**DSA for Mechatronics · Week 03**

> **Your task:** Run every cell from top to bottom.  
> At the end, a full report is printed automatically.  
> **Submit:** A screenshot or PDF of the final report cell output.  
> **Present:** Be ready to explain what each step does and why.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ║   This seeds your personal dataset — be honest!     ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

# ── seed everything from your ID ──────────────────────
import random, hashlib
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready — run the next cells.")


## Step 1 — Generate your personal PWM signal

In [ ]:
# Servo controller PWM signal — personalised thresholds and weights
N = random.randint(250, 400)

# Each student gets slightly different thresholds and command distribution
REVERSE_MAX = random.randint(1100, 1250)
FORWARD_MIN = random.randint(1750, 1900)
WEIGHTS     = [random.randint(1, 4), random.randint(2, 6), random.randint(2, 5)]

# Hidden calibration sequence (5 pulses) — unique per student
SEQ_LEN      = 5
START_SEQ    = [random.choice([900, 950, 1000]) if i % 2 == 0
                else random.choice([2000, 2050, 2100])
                for i in range(SEQ_LEN)]

def random_pulse():
    cmd = random.choices(['REVERSE', 'STOP', 'FORWARD'], weights=WEIGHTS)[0]
    if cmd == 'REVERSE': return random.randint(900, REVERSE_MAX - 1)
    if cmd == 'STOP':    return random.randint(REVERSE_MAX, FORWARD_MIN)
    return random.randint(FORWARD_MIN + 1, 2100)

pwm_signal = [random_pulse() for _ in range(N)]
inject_pos = random.randint(10, N - SEQ_LEN - 5)
for off, val in enumerate(START_SEQ):
    pwm_signal[inject_pos + off] = val

print("PWM signal parameters:")
print(f"  Total pulses       : {N}")
print(f"  REVERSE if pulse < : {REVERSE_MAX} µs")
print(f"  FORWARD if pulse > : {FORWARD_MIN} µs")
print(f"  Command weights    : REV={WEIGHTS[0]}  STOP={WEIGHTS[1]}  FWD={WEIGHTS[2]}")
print(f"  Start sequence     : {START_SEQ}  (hidden somewhere)")
print()
print(f"First 12 pulses (µs):")
for i, p in enumerate(pwm_signal[:12]):
    print(f"  [{i:03d}] {p} µs")


## Step 2 — Decode pulse widths to commands (list comprehension)

In [ ]:
def decode_signal(pwm, reverse_max, forward_min):
    """Decode each pulse to REVERSE / STOP / FORWARD based on thresholds."""
    return [
        'REVERSE' if p < reverse_max else ('FORWARD' if p > forward_min else 'STOP')
        for p in pwm
    ]

commands = decode_signal(pwm_signal, REVERSE_MAX, FORWARD_MIN)

print(f"Decoded {len(commands)} commands:")
print()
print(f"  {'Index':>6}  {'Pulse (µs)':>11}  Command")
print(f"  {'─'*6}  {'─'*11}  {'─'*10}")
for i in range(15):
    print(f"  [{i:03d}]  {pwm_signal[i]:>10}  {commands[i]}")


## Step 3 — Count command frequencies

In [ ]:
def count_commands(commands):
    """Build a frequency map: {command: count}."""
    freq = {}
    for cmd in commands:
        freq[cmd] = freq.get(cmd, 0) + 1
    return freq

freq  = count_commands(commands)
total = len(commands)

print(f"Command frequency table (total: {total}):")
print()
bar_max = 40
for cmd, count in sorted(freq.items(), key=lambda x: -x[1]):
    pct = count / total * 100
    bar = '█' * int(pct / 100 * bar_max)
    print(f"  {cmd:<10}: {count:>5}  ({pct:5.1f}%)  {bar}")


## Step 4 — Find the longest consecutive command run (two pointers)

In [ ]:
def longest_run(commands):
    """Return (command, start_index, length) of the longest uninterrupted run."""
    best_cmd, best_start, best_len = commands[0], 0, 1
    run_start = 0
    for i in range(1, len(commands)):
        if commands[i] != commands[run_start]:
            if i - run_start > best_len:
                best_cmd, best_start, best_len = commands[run_start], run_start, i - run_start
            run_start = i
    if len(commands) - run_start > best_len:
        best_cmd, best_start, best_len = commands[run_start], run_start, len(commands) - run_start
    return best_cmd, best_start, best_len

run_cmd, run_start, run_len = longest_run(commands)

print(f"Longest consecutive command run:")
print(f"  Command    : {run_cmd}")
print(f"  Start index: {run_start}")
print(f"  Length     : {run_len} pulses")
print(f"  Pulse range: {min(pwm_signal[run_start:run_start+run_len])} – "
      f"{max(pwm_signal[run_start:run_start+run_len])} µs")


## Step 5 — Locate the hidden calibration sequence (pattern matching)

In [ ]:
def find_sequence(pwm, pattern):
    """Return the first index where pattern appears in pwm, or -1."""
    n, k = len(pwm), len(pattern)
    for i in range(n - k + 1):
        if pwm[i : i + k] == pattern:
            return i
    return -1

found = find_sequence(pwm_signal, START_SEQ)

print(f"Searching for start sequence: {START_SEQ}")
print()
if found >= 0:
    print(f"  ✅ Found at index {found}")
    print(f"  Signal values  : {pwm_signal[found : found + SEQ_LEN]}")
    print(f"  Decoded cmds   : {decode_signal(pwm_signal[found:found+SEQ_LEN], REVERSE_MAX, FORWARD_MIN)}")
else:
    print("  ❌ Not found")


## 📋 Final Report — this is what you submit

In [ ]:
dominant_cmd = max(freq, key=freq.get)
W = 56
print("╔" + "═"*W + "╗")
print("║" + " PWM SIGNAL DECODER — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<24}: {STUDENT_ID:<28}║")
print(f"║  {'Dataset seed':<24}: {_seed:<28}║")
print("╠" + "═"*W + "╣")
print("║  SIGNAL PARAMETERS" + " "*(W-19) + "║")
print(f"║  {'Total pulses':<24}: {N:<28}║")
print(f"║  {'REVERSE threshold':<24}: < {REVERSE_MAX} µs{'':<23}║")
print(f"║  {'FORWARD threshold':<24}: > {FORWARD_MIN} µs{'':<23}║")
print("╠" + "═"*W + "╣")
print("║  COMMAND FREQUENCIES" + " "*(W-21) + "║")
for cmd, count in sorted(freq.items(), key=lambda x: -x[1]):
    pct = count / total * 100
    tag = "  ← dominant" if cmd == dominant_cmd else ""
    print(f"║  {cmd:<10}  {count:>5}  ({pct:5.1f}%){tag:<{W-27}}║")
print("╠" + "═"*W + "╣")
print("║  LONGEST RUN" + " "*(W-13) + "║")
print(f"║  {'Command':<24}: {run_cmd:<28}║")
print(f"║  {'Start index':<24}: {run_start:<28}║")
print(f"║  {'Length':<24}: {run_len} pulses{'':<21}║")
print("╠" + "═"*W + "╣")
print("║  CALIBRATION SEQUENCE" + " "*(W-22) + "║")
print(f"║  {'Pattern':<24}: {str(START_SEQ):<28}║")
seq_result = f"found at index {found}" if found >= 0 else "not found"
print(f"║  {'Result':<24}: {seq_result:<28}║")
print("╚" + "═"*W + "╝")
